# DIET Path 2 — India Tech Career Intelligence (2026)
## Exploratory Data Analysis & Workforce Intelligence
**Event:** Manthan Hackathon — Calibo AI Academy  
**Institution:** Dhanekula Institute of Engineering & Technology (DIET)  
**Date:** Saturday, 31 October 2026  
**Technology Stack:** Python, Pandas, NumPy, Matplotlib, Seaborn

### Team Members:
1. **Anne Mukesh Balaji**
2. **Kesiboyina Chandrika**
3. **Chennam Divya**
4. **Chinnam Joshitha**

---
## Problem Statement 1: Audit and Clean the Career Dataset
**Execution Window:** 09:50 AM – 10:45 AM (IST)

### 1. Import Libraries
Importing required libraries for data manipulation and visualization.

In [22]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Visual configuration for charts
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["font.size"] = 11

### 2. Load and Understand the Dataset
Loading the raw dataset  and inspecting structure, data types, and memory usage.

In [23]:
df = pd.read_csv("indian_tech_career_intelligence_2026.csv")
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 33 columns):
 #   Column                Non-Null Count   Dtype
---  ------                --------------   -----
 0   job_title             100000 non-null  str  
 1   job_category          100000 non-null  str  
 2   company_name          100000 non-null  str  
 3   company_size          100000 non-null  str  
 4   industry              100000 non-null  str  
 5   company_type          100000 non-null  str  
 6   location_city         100000 non-null  str  
 7   location_state        100000 non-null  str  
 8   location_country      100000 non-null  str  
 9   work_mode             100000 non-null  str  
 10  salary_minimum        100000 non-null  int64
 11  salary_maximum        100000 non-null  int64
 12  salary_currency       100000 non-null  str  
 13  stipend_amount        100000 non-null  int64
 14  experience_level      100000 non-null  str  
 15  experience_required   100000 non-null  str  
 

### 3. Step 1: Duplicate Rows Audit
Auditing the dataset for duplicate records across all 33 columns.

In [24]:
num_duplicates = df.duplicated().sum()
print(f"Duplicate Rows: {num_duplicates}")

Duplicate Rows: 0


> **Audit Finding:** There are **0 duplicate rows** in the dataset. All 100,000 listings are unique.

### 4. Step 2: Missing Values Audit
Checking missing value counts and percentages across all 33 attributes.

In [25]:
missing = df.isnull().sum()
missing_cols = missing[missing > 0]
print(missing_cols)

probation_period    33499
dtype: int64


> **Audit Finding:** The `probation_period` column is the only field with missing values (**33,499 rows**, or **33.5%**). All other 32 columns are 100% complete.

### 5. Step 3: Handling Missing Values in `probation_period`

**Observed Distribution:**
* **NaN (Missing):** 33,499 rows (33.5%)
* **6 Months:** 33,443 rows (33.4%)
* **3 Months:** 33,058 rows (33.1%)

**Why we do NOT impute with Mode ("6 Months"):**
If we replace NaN with "6 Months", we would be guessing and fabricating data for 33,500 listings! The hackathon guidelines explicitly state: *"Do not hide missing values or data issues; document them."*

**Why we are NOT dropping the column:**
1. Problem Statement 7 explicitly requires analyzing `probation_period` category counts and missingness.
2. 66.5% (over 66,500 listings) contain valid, valuable data that should not be discarded.
3. In job market operations, missing probation clauses carry real business meaning (e.g., contract roles or startups without formal probation policies).

**Why we are NOT dropping the 33,499 rows:**
Dropping 33.5% of the dataset would discard valid information for all other 32 columns.

**Final Decision:**
Since `probation_period` is a categorical column, we impute missing values with the global constant **`'Not Specified'`** (Missing-as-a-Category Encoding). This preserves the sample size, avoids introducing bias, and ensures 100% transparency.

In [26]:
df["probation_period"] = df["probation_period"].fillna("Not Specified")

print("Updated probation_period distribution:")
print(df["probation_period"].value_counts(dropna=False))

Updated probation_period distribution:
probation_period
Not Specified    33499
6 Months         33443
3 Months         33058
Name: count, dtype: int64


### 6. Step 4: Text Normalization (Whitespace Stripping)
Strip leading and trailing whitespace from all string/object columns to prevent accidental duplicate categories (e.g.,  vs ).

In [27]:
string_cols = df.select_dtypes(include=["object", "str"]).columns
for col in string_cols:
    df[col] = df[col].astype(str).str.strip()

print(f"✓ Stripped whitespace across {len(string_cols)} text columns.")

✓ Stripped whitespace across 27 text columns.


### 7. Step 5: Datetime Parsing & Timeline Validation
1. Convert  and  to official datetime objects using Day-Month-Year format ().
2. Audit for malformed/unparseable dates.
3. **Timeline Sanity Check:** Verify that no  occurs before its .

In [28]:
df["posted_date"] = pd.to_datetime(
    df["posted_date"], format="%d-%m-%Y", errors="coerce"
)
df["application_deadline"] = pd.to_datetime(
    df["application_deadline"], format="%d-%m-%Y", errors="coerce"
)

unparseable_posted = df["posted_date"].isnull().sum()
unparseable_deadline = df["application_deadline"].isnull().sum()
print(f"Unparseable 'posted_date' entries: {unparseable_posted}")
print(f"Unparseable 'application_deadline' entries: {unparseable_deadline}")

# Sanity check: deadline >= posted_date
invalid_timeline = (df["application_deadline"] < df["posted_date"]).sum()
print(f"Invalid timeline records (deadline < posted date): {invalid_timeline}")

print(f"Date Range:")
print(f"• Earliest Posted: {df['posted_date'].min().strftime('%d-%b-%Y')}")
print(f"• Latest Posted:   {df['posted_date'].max().strftime('%d-%b-%Y')}")

Unparseable 'posted_date' entries: 0
Unparseable 'application_deadline' entries: 0
Invalid timeline records (deadline < posted date): 0
Date Range:
• Earliest Posted: 01-Jan-2026
• Latest Posted:   30-May-2026


### 8. Step 6 & 7: Numeric Conversion & Compensation Sanity Checks

1. **Clean :**
   - Stored as text with '%' symbol (e.g. '85%').
   - Stripped '%' and cast to  () to enable statistical aggregation.
   - *Caveat:* The metric methodology is not specified by the data provider, so we treat it as descriptive metadata, not a guaranteed prediction.

2. **Validate Compensation Fields:**
   - Confirm currency is strictly in .
   - Verify  (no inverted ranges).
   - Check that there are no negative values in salary or stipend amounts.

In [29]:
# 1. Clean career_growth_index: strip '%' and cast to float
df["career_growth_index_pct"] = (
    df["career_growth_index"]
    .str.rstrip("%")
    .astype(float)
)

print("=== CAREER GROWTH INDEX SUMMARY ===")
print(df["career_growth_index_pct"].describe().round(2))

# 2. Validate Salary Currency
print("=== SALARY CURRENCY AUDIT ===")
print(df["salary_currency"].value_counts())

# 3. Sanity check: Ensure salary_minimum <= salary_maximum
invalid_salary_order = (df["salary_minimum"] > df["salary_maximum"]).sum()
print(f"Inverted salary ranges (min > max): {invalid_salary_order}")

# 4. Check for negative numbers
negative_salaries = (df["salary_minimum"] < 0).sum() + (df["salary_maximum"] < 0).sum()
negative_stipends = (df["stipend_amount"] < 0).sum()
print(f"Negative salary values:  {negative_salaries}")
print(f"Negative stipend values: {negative_stipends}")

# 5. Compensation Summary
print("=== COMPENSATION RANGES (INR) ===")
print(f"• Salary Minimum Range: ₹{df['salary_minimum'].min():,} to ₹{df['salary_minimum'].max():,}")
print(f"• Salary Maximum Range: ₹{df['salary_maximum'].min():,} to ₹{df['salary_maximum'].max():,}")
print(f"• Internship Stipend:   ₹{df['stipend_amount'].min():,} to ₹{df['stipend_amount'].max():,}")

=== CAREER GROWTH INDEX SUMMARY ===
count    100000.00
mean         81.51
std           9.82
min          65.00
25%          73.00
50%          82.00
75%          90.00
max          98.00
Name: career_growth_index_pct, dtype: float64
=== SALARY CURRENCY AUDIT ===
salary_currency
INR    100000
Name: count, dtype: int64
Inverted salary ranges (min > max): 0
Negative salary values:  0
Negative stipend values: 0
=== COMPENSATION RANGES (INR) ===
• Salary Minimum Range: ₹300,004 to ₹599,999
• Salary Maximum Range: ₹700,013 to ₹1,799,996
• Internship Stipend:   ₹15,000 to ₹44,999


### 9. Step 8: Export Cleaned Dataset
Exporting the cleaned DataFrame to  while keeping the raw file untouched.

In [30]:
cleaned_file_path = "indian_tech_career_intelligence_2026_cleaned.csv"
df.to_csv(cleaned_file_path, index=False)

print(f"✓ Cleaned dataset saved successfully as: '{cleaned_file_path}'")
print(f"Total rows exported: {len(df):,}")
print(f"Total columns exported: {df.shape[1]}")

✓ Cleaned dataset saved successfully as: 'indian_tech_career_intelligence_2026_cleaned.csv'
Total rows exported: 100,000
Total columns exported: 34


---
### Official Data Cleaning Decision Log (Problem Statement 1)

| Field / Attribute | Audit Finding | Action Applied | Analytical & Business Justification |
| :--- | :--- | :--- | :--- |
| **Entire Dataset** | Raw data preservation required | Exported cleaned copy to separate CSV | Ensures reproducibility and raw data safety per hackathon rules. |
| **Duplicate Rows** | Evaluated `.duplicated().sum()` | Confirmed 0 duplicates; 100k rows preserved | Confirms every job posting record is distinct. |
| **`probation_period`** | 33,499 missing values (33.5%) | Imputed with global constant `'Not Specified'` | Preserves all rows and column for PS 7; avoids mode imputation bias ("6 Months"). |
| **String Columns (27 cols)** | Potential leading/trailing spaces | Applied `.str.strip()` | Eliminates category duplication caused by accidental whitespace. |
| **`posted_date` & `application_deadline`** | Stored as text in `DD-MM-YYYY` format | Parsed with `pd.to_datetime(format='%d-%m-%Y')` | Standardizes date objects; verified 0 deadline ordering anomalies. |
| **`career_growth_index`** | Stored as string with `%` symbol | Stripped `%` and cast to float `85.0` | Enables statistical summary (mean: 81.51%, median: 82.0%). |
| **Salary & Stipends** | Compensation bounds audit | Checked `min <= max`, INR currency, non-negativity | 100% currency is INR; 0 inverted ranges; 0 negative values. |
